In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Run this cell only if executing on a remote Colab kernel
if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ:
    repo_dir = "/content/walmart-sales-forecasting"
    repo_url = "https://github.com/NikaMikeltadze/walmart-sales-forecasting.git"

    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", repo_url, repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull"], check=True)

    # Pin everyone's Colab runtime to the same library versions as the project's requirements.txt
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", os.path.join(repo_dir, "requirements.txt"), "--quiet"],
        check=True,
    )

    os.chdir(os.path.join(repo_dir, "notebooks"))

    if repo_dir not in sys.path:
        sys.path.append(repo_dir)

    # data/raw/ is gitignored, so the clone above has code but not the competition CSVs.
    # Upload train.csv/, test.csv/, features.csv/, stores.csv, sampleSubmission.csv to this
    # folder in your Google Drive first: MyDrive/walmart-sales-forecasting/data/raw
    from google.colab import drive

    drive.mount("/content/drive")

    drive_data_dir = Path("/content/drive/MyDrive/walmart-sales-forecasting/data/raw")
    local_data_dir = Path(repo_dir) / "data" / "raw"

    if not (local_data_dir / "train.csv").exists():
        if not drive_data_dir.exists():
            raise FileNotFoundError(
                f"Expected competition data in Google Drive at {drive_data_dir}. "
                "Upload train.csv/, test.csv/, features.csv/, stores.csv, and "
                "sampleSubmission.csv there, then re-run this cell."
            )
        local_data_dir.mkdir(parents=True, exist_ok=True)
        for item in drive_data_dir.iterdir():
            dest = local_data_dir / item.name
            if dest.exists():
                continue
            if item.is_dir():
                shutil.copytree(item, dest)
            else:
                shutil.copy2(item, dest)

# TimesFM - Walmart Store Sales Forecasting (bonus)

**Zero-shot forecasting with a pretrained time-series foundation model.**

TimesFM is Google's pretrained time-series foundation model. Unlike every other
model in this project, it does **no training and no feature engineering** here: we
feed each `(Store, Dept)` series' past weekly `Weekly_Sales` straight into the
pretrained model and read off a multi-week forecast. This notebook is the optional
bonus from `docs/person_a_prompt.md` - the point is to see how a pretrained model
does on this task out of the box, next to the tuned tree models and ARIMA.

It mirrors the ARIMA notebook's structure: a plain `.fit`/`.predict` wrapper
(`src/timesfm_forecaster.py`), the same shared holdout fold for an honest WMAE, and
a full-coverage Kaggle submission with a mean-fallback cascade so no test row is NaN.

> Designed for a **Colab GPU runtime**. On CPU it runs but is slow. TimesFM is
> installed by the next cell (not in `requirements.txt`, since it is heavy and its
> API is version-sensitive).

In [ ]:
# TimesFM install - kept OUT of requirements.txt on purpose (heavy: pulls torch
# plus a ~500M-param Hugging Face checkpoint, and the API is version-sensitive).
# Runs on Colab or whenever `import timesfm` is not available. Meant for a Colab
# GPU runtime; on CPU it still runs but is slow.
import importlib
import subprocess
import sys


def _available(module_name):
    try:
        importlib.import_module(module_name)
        return True
    except Exception:
        return False


if not _available("timesfm"):
    print("Installing timesfm (torch backend) - this can take a few minutes ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "timesfm[torch]"],
        check=True,
    )
    print("Done. If the import still fails below, restart the runtime and re-run.")
else:
    print("timesfm already installed.")

In [ ]:
import os
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import warnings

import joblib
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd

from src.timesfm_forecaster import TimesFMForecaster
from src.evaluation import weighted_mae
from src.preprocessing import load_raw_data
from src.validation import describe_split, expanding_window_splits, split_frame

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

DATA_DIR = Path.cwd().parent / "data" / "raw"
SUBMISSIONS_DIR = Path.cwd().parent / "submissions"
SUBMISSIONS_DIR.mkdir(exist_ok=True)

M = 52                # weekly seasonal period (yearly seasonality)
CONTEXT_LEN = 512     # weeks of history fed to TimesFM (>= longest series, ~143)
HORIZON = 39          # full Kaggle test span (2012-11-02 -> 2013-07-26)
CHECKPOINT = "google/timesfm-2.0-500m-pytorch"
# "gpu" on a Colab GPU runtime; "cpu" everywhere else.
BACKEND = "gpu" if ("COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ) else "cpu"
print("TimesFM backend:", BACKEND)

## MLflow setup

###  Manual credentials override (VS Code / non-Colab-UI runtimes)

In [ ]:
import os
from getpass import getpass

# Only prompt for values not already set, so re-running cells doesn't re-ask.
# Leave a prompt blank to fall through to Colab Secrets / .env in the next cell.
if not os.environ.get("MLFLOW_TRACKING_USERNAME"):
    _user = getpass("DagsHub username (blank -> use Colab Secrets / .env): ").strip()
    if _user:
        os.environ["MLFLOW_TRACKING_USERNAME"] = _user
if not os.environ.get("MLFLOW_TRACKING_PASSWORD"):
    _token = getpass("DagsHub token (blank -> use Colab Secrets / .env): ").strip()
    if _token:
        os.environ["MLFLOW_TRACKING_PASSWORD"] = _token

In [ ]:
def _load_env_file(path):
    """Minimal .env loader (KEY=VALUE per line) - avoids adding python-dotenv
    as a dependency for a two-line credentials file. Uses setdefault so an
    env var already set in the shell (local dev) is never overridden."""
    path = Path(path)
    if not path.exists():
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        os.environ.setdefault(key.strip(), value.strip())


if os.environ.get("MLFLOW_TRACKING_USERNAME") and os.environ.get("MLFLOW_TRACKING_PASSWORD"):
    # Already provided by the manual-override cell above (e.g. driving a Colab
    # runtime from VS Code, where google.colab.userdata would time out). Do NOT
    # evaluate userdata.get(...) in that case - it blocks and then raises.
    pass
elif "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ:
    # Colab Secrets (key icon, left sidebar) - each teammate adds their own
    # DAGSHUB_USERNAME/DAGSHUB_TOKEN secret, tied to their own Google account.
    # Never stored in the notebook or repo, unlike a hardcoded token would be.
    from google.colab import userdata

    os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("DAGSHUB_USERNAME")
    os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")
else:
    _load_env_file(Path.cwd().parent / ".env")

In [ ]:
MLFLOW_TRACKING_URI = "https://dagshub.com/NikaMikeltadze/walmart-sales-forecasting.mlflow"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

try:
    mlflow.set_experiment("TimesFM_Training")
    mlflow.MlflowClient().search_experiments(max_results=1)  # force a network round trip now
except Exception as e:
    raise RuntimeError(
        "Could not authenticate to the DagsHub MLflow server at "
        f"{MLFLOW_TRACKING_URI}. Set MLFLOW_TRACKING_USERNAME and "
        "MLFLOW_TRACKING_PASSWORD (a DagsHub access token) as environment "
        "variables, then re-run this cell. Not falling back to local "
        "./mlruns - that would desync from Person B's runs."
    ) from e

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Active experiment:", mlflow.get_experiment_by_name("TimesFM_Training").name)

## Load raw data

In [ ]:
raw = load_raw_data(DATA_DIR)
for name, df in raw.items():
    print(f"{name}: {df.shape}")

assert raw["train"].shape[0] == 421_570, "unexpected train row count - check data/raw"
assert raw["test"].shape[0] > 0
assert raw["stores"].shape[0] == 45

## How TimesFM is used here (zero-shot)

TimesFM is a decoder-style transformer pretrained on a large corpus of real and
synthetic time series. It maps a context window of past values to a forecast of the
next H steps, with **no per-series fitting** - the same frozen weights forecast every
series.

For this task, per `(Store, Dept)` series:

- **Context**: the last `CONTEXT_LEN` (512) weeks of `Weekly_Sales`, reindexed onto a
  regular weekly grid with small internal gaps filled. A full-length series has ~143
  weeks, comfortably more than one 52-week cycle, so the model can pick up the yearly
  retail seasonality on its own.
- **Horizon**: 39 weeks for the Kaggle test span (13 weeks when scoring on the shared
  holdout fold). Test weeks are contiguous immediately after train's end, so a single
  forward forecast lines up with the test dates by position.
- **Frequency**: `freq=1` (TimesFM's medium-frequency bucket for weekly/monthly data).

**Expected weakness**: 39 weeks is a long horizon relative to typical foundation-model
context, so the far end of the forecast is the least reliable part. Series shorter than
`min_context` weeks, or `(Store, Dept)` pairs unseen in train, fall back to a store/dept
mean so every one of the 115,064 test rows still gets a finite prediction.

## Build the series to model

The company-level aggregate and 3 representative high-volume `(Store, Dept)` series -
the **same** representative set as the ARIMA notebook, so the diagnostic numbers are
directly comparable.

In [ ]:
train = raw["train"].copy()
train["Date"] = pd.to_datetime(train["Date"])

# Company-level weekly total sales (single aggregate series).
agg = train.groupby("Date")["Weekly_Sales"].sum().sort_index()

# Per-week holiday flag, shared by the aggregate scoring below.
holiday_by_date = train.groupby("Date")["IsHoliday"].any()

# 3 representative high-volume (Store, Dept) series with full-length history.
series_len = train.groupby(["Store", "Dept"])["Date"].nunique()
full_len = series_len.max()
eligible = series_len[series_len == full_len].index
series_means = train.groupby(["Store", "Dept"])["Weekly_Sales"].mean()
rep_keys = series_means.loc[eligible].sort_values(ascending=False).head(3).index.tolist()
rep_keys = [(int(s), int(d)) for s, d in rep_keys]


def build_series(store, dept):
    """Return (values Series, holiday Series) both indexed by Date for one series."""
    g = train[(train["Store"] == store) & (train["Dept"] == dept)].sort_values("Date")
    values = g.set_index("Date")["Weekly_Sales"]
    values = values[~values.index.duplicated(keep="last")]
    holiday = g.groupby("Date")["IsHoliday"].any().reindex(values.index).fillna(False)
    return values, holiday


print("Aggregate series:", agg.shape[0], "weeks,",
      agg.index.min().date(), "->", agg.index.max().date())
print("Representative (Store, Dept) series:", rep_keys)
agg.tail(3)

## Time-based validation window

The shared expanding-window CV splitter, same folds every other model uses. We report
on the last fold (`HOLDOUT`) so WMAE is comparable across all architectures.

In [ ]:
splits = expanding_window_splits(pd.Series(agg.index), n_splits=3, val_weeks=13)
assert len(splits) == 3, "history too short for 3 folds - check the data"
HOLDOUT = splits[-1]
N_VAL = (HOLDOUT.val_end - HOLDOUT.val_start).days // 7 + 1
for i, s in enumerate(splits):
    print(f"fold {i}:", describe_split(s))
print("Holdout fold:", describe_split(HOLDOUT), "| N_VAL =", N_VAL)

## Zero-shot diagnostic (get the plumbing working)

Load TimesFM once and zero-shot the company aggregate + the 3 representative series
over the holdout window. This is the quick "does it work and how does it look" step,
logged as `TimesFM_ZeroShot`. The aggregate is on a company-wide scale (~tens of
millions) and the per-series numbers are per-department, so they are reported as
separate WMAEs rather than pooled.

In [ ]:
# Forecast the aggregate + rep series over the holdout window, all in one batch.
diag = TimesFMForecaster(context_len=CONTEXT_LEN, horizon_len=N_VAL,
                         checkpoint=CHECKPOINT, backend=BACKEND, freq=1)

val_mask = (holiday_by_date.index >= HOLDOUT.val_start) & (holiday_by_date.index <= HOLDOUT.val_end)
val_dates = holiday_by_date.index[val_mask]
val_holiday_flags = holiday_by_date[val_mask].to_numpy()

diag_series = [("aggregate", agg)] + [(f"{s}_{d}", build_series(s, d)[0]) for (s, d) in rep_keys]
contexts, actuals, names = [], [], []
for name, ser in diag_series:
    ser = ser[~ser.index.duplicated(keep="last")].sort_index()
    contexts.append(ser[ser.index <= HOLDOUT.train_end].to_numpy(dtype=float)[-CONTEXT_LEN:])
    actuals.append(ser.reindex(val_dates).to_numpy(dtype=float))
    names.append(name)

forecasts = diag._forecast_batch(contexts)

per_series = {}
for name, act, fc in zip(names, actuals, forecasts):
    fc = np.clip(np.asarray(fc, dtype=float)[:len(act)], 0.0, None)
    ok = np.isfinite(act)
    per_series[name] = float(weighted_mae(act[ok], fc[ok], val_holiday_flags[ok]))

agg_wmae = per_series["aggregate"]
rep_wmae_mean = float(np.mean([v for k, v in per_series.items() if k != "aggregate"]))
print("Per-series zero-shot WMAE:", {k: round(v, 1) for k, v in per_series.items()})
print("Aggregate WMAE:", round(agg_wmae, 2), "| representative-series mean WMAE:", round(rep_wmae_mean, 2))

# Forecast-vs-actual plot on the aggregate for the artifact.
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(agg.index[-104:], agg.to_numpy()[-104:], label="history", color="steelblue")
ax.plot(val_dates, actuals[0], label="actual (holdout)", color="black")
ax.plot(val_dates, np.clip(np.asarray(forecasts[0], dtype=float)[:len(val_dates)], 0, None),
        label="TimesFM zero-shot", color="crimson", linestyle="--")
ax.axvline(HOLDOUT.train_end, color="gray", linestyle=":")
ax.set_title("TimesFM zero-shot - company aggregate over the holdout window")
ax.legend()
fig.tight_layout()
DIAG_PLOT = SUBMISSIONS_DIR / "_timesfm_zeroshot_aggregate.png"
fig.savefig(DIAG_PLOT, dpi=110)
plt.close(fig)

with mlflow.start_run(run_name="TimesFM_ZeroShot"):
    mlflow.set_tag("experiment_group", "diagnostic")
    mlflow.log_param("scope", "company_aggregate + 3 representative series")
    mlflow.log_param("checkpoint", CHECKPOINT)
    mlflow.log_param("context_len", CONTEXT_LEN)
    mlflow.log_param("horizon_len", N_VAL)
    mlflow.log_param("freq", 1)
    mlflow.log_params(describe_split(HOLDOUT))
    for name, v in per_series.items():
        mlflow.log_metric(f"wmae_{name}", v)
    mlflow.log_metric("wmae_aggregate", agg_wmae)
    mlflow.log_metric("wmae_representative_series_mean", rep_wmae_mean)
    mlflow.log_artifact(str(DIAG_PLOT))

## Validating the full-panel strategy on the shared holdout

The honest number: fit the forecaster on all history up to the holdout `train_end`,
zero-shot every `(Store, Dept)` series over the 13-week validation window, and score
WMAE across the whole panel - directly comparable to the tree models' and ARIMA's
holdout WMAE.

In [ ]:
train = raw["train"].copy()
train["Date"] = pd.to_datetime(train["Date"])
train_holdout, val_holdout = split_frame(train, HOLDOUT)

holdout_forecaster = TimesFMForecaster(
    context_len=CONTEXT_LEN, horizon_len=N_VAL,
    checkpoint=CHECKPOINT, backend=BACKEND, freq=1, verbose=True,
)
holdout_forecaster.fit(train_holdout)
holdout_preds = holdout_forecaster.predict(val_holdout[["Store", "Dept", "Date"]])

y_true = val_holdout["Weekly_Sales"].to_numpy()
final_strategy_wmae = weighted_mae(y_true, holdout_preds, val_holdout["IsHoliday"].to_numpy())
final_strategy_mae = float(np.mean(np.abs(y_true - holdout_preds)))
print("Full-panel zero-shot holdout WMAE:", round(final_strategy_wmae, 2),
      "| MAE:", round(final_strategy_mae, 2))

with mlflow.start_run(run_name="TimesFM_Final_HoldoutValidation"):
    mlflow.set_tag("experiment_group", "final_validation")
    mlflow.log_param("strategy", "zero_shot_timesfm_per_series")
    mlflow.log_param("checkpoint", CHECKPOINT)
    mlflow.log_param("context_len", CONTEXT_LEN)
    mlflow.log_param("horizon_len", N_VAL)
    mlflow.log_param("freq", 1)
    mlflow.log_param("min_context", holdout_forecaster.min_context)
    mlflow.log_params(describe_split(HOLDOUT))
    mlflow.log_metric("wmae", final_strategy_wmae)
    mlflow.log_metric("mae", final_strategy_mae)

## Final fit + full-coverage forecaster

Fit on all of train (caches per-series history) and pickle the forecaster. The pickle
stores only the cached history and config - the TimesFM weights come from the Hugging
Face checkpoint, not the pickle. Headline `wmae`/`mae` on `TimesFM_Final` are the
honest holdout numbers from the previous cell.

In [ ]:
forecaster = TimesFMForecaster(
    context_len=CONTEXT_LEN, horizon_len=HORIZON,
    checkpoint=CHECKPOINT, backend=BACKEND, freq=1, verbose=True,
)
forecaster.fit(raw["train"])

MODEL_PATH = SUBMISSIONS_DIR / "_timesfm_forecaster.joblib"
joblib.dump(forecaster, MODEL_PATH)  # history + config only; weights come from the HF checkpoint

with mlflow.start_run(run_name="TimesFM_Final") as final_run:
    mlflow.set_tag("experiment_group", "final")
    mlflow.log_param("strategy", "zero_shot_timesfm_per_series")
    mlflow.log_param("checkpoint", CHECKPOINT)
    mlflow.log_param("context_len", CONTEXT_LEN)
    mlflow.log_param("horizon_len", HORIZON)
    mlflow.log_param("backend", BACKEND)
    mlflow.log_param("freq", 1)
    mlflow.log_param("min_context", forecaster.min_context)
    mlflow.log_param("n_series", len(forecaster.history_))
    mlflow.log_metric("wmae", final_strategy_wmae)  # honest holdout number
    mlflow.log_metric("mae", final_strategy_mae)
    mlflow.log_metric("reference_zeroshot_aggregate_wmae", agg_wmae)
    mlflow.log_metric("reference_zeroshot_representative_series_wmae_mean", rep_wmae_mean)
    mlflow.log_artifact(str(MODEL_PATH))
    final_run_id = final_run.info.run_id
print("TimesFM_Final run:", final_run_id)

## Submission

Forecast every row of raw `test.csv` (39-week horizon) and write the Kaggle submission.
Same hard integrity checks as the other models; no test row is ever NaN.

In [ ]:
test_preds = forecaster.predict(raw["test"])
assert not np.isnan(test_preds).any(), "forecaster produced NaN predictions on raw test data"

submission = pd.DataFrame(
    {
        "Id": (
            raw["test"]["Store"].astype(str) + "_"
            + raw["test"]["Dept"].astype(str) + "_"
            + pd.to_datetime(raw["test"]["Date"]).dt.strftime("%Y-%m-%d")
        ),
        "Weekly_Sales": test_preds,
    }
)

SUBMISSION_PATH = SUBMISSIONS_DIR / "timesfm_submission.csv"
submission.to_csv(SUBMISSION_PATH, index=False)

# Hard integrity checks (do not depend on any other file).
assert submission.shape == (115064, 2), submission.shape
assert list(submission.columns) == ["Id", "Weekly_Sales"]
assert not submission["Weekly_Sales"].isna().any()

# Optional row-for-row cross-check against sampleSubmission.
try:
    sample = pd.read_csv(DATA_DIR / "sampleSubmission.csv")
    assert submission["Id"].tolist() == sample["Id"].tolist(), "Id order mismatch vs sampleSubmission"
    print("Submission matches sampleSubmission row-for-row:", submission.shape[0], "rows, no NaNs.")
except (FileNotFoundError, PermissionError) as e:
    print("Skipped sampleSubmission cross-check:", type(e).__name__)
submission.head()

In [ ]:
with mlflow.start_run(run_id=final_run_id):
    mlflow.log_artifact(str(SUBMISSION_PATH))
print("Logged submission to the TimesFM_Final run.")